# Etapa 3 — Smoke Multicaso y Multi-Semilla (`smoke_all_cases_campaign`)

Replica `tools/smoke_all_cases_campaign.py` ejecutando **únicamente los modelos promovidos** desde `promoted_models.json`:
- En `full_campaign` recorre **todo el catálogo (29 casos)**: `benchmarks` + `thesis_curated` (`xlpe_*`, `aras_*`, `coaxial_*`, `kim_*`, `symmetric_*`). En `demo_fast` evalúa el subconjunto pivote de cables y MMS (`mms_constant`, `mms_layered_y`, `cable_disk`, `xlpe_single`, `xlpe_discrete_layers`, `aras_flat`, `coaxial_robin`).
- Evalúa variabilidad entre semillas (`SEEDS = (0, 1, 2)` en `full_campaign`; `(0,)` en `demo_fast`) en `float32` con `adam_steps = 100`, `lbfgs_steps = 10`, caché **FEM WSL2** y reanudación opcional (`resume_from`).

In [1]:
import json
import os
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "full_campaign"    #"demo_fast"

# Modo de sobreescritura / reutilización desde disco:
# - OVERWRITE = False -> carga resultados ya guardados en disco y solo ejecuta corridas faltantes
# - OVERWRITE = True  -> ignora caché previa y vuelve a entrenar/calcular todo desde cero
OVERWRITE = True

# Paralelismo de campaña:
# - USE_PARALLEL=True  -> fuerza device='cpu' con ParallelRunner multi-proceso
# - USE_PARALLEL=False -> ejecuta secuencialmente en device='auto' (p. ej. XPU nativo sin fallbacks)
USE_PARALLEL = True
MAX_WORKERS = min(12, os.cpu_count() or 1)
DEVICE = "cpu" if USE_PARALLEL else "auto"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | OVERWRITE={OVERWRITE} | USE_PARALLEL={USE_PARALLEL} (workers={MAX_WORKERS}, device={DEVICE!r}) | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='full_campaign' | OVERWRITE=True | USE_PARALLEL=True (workers=12, device='cpu') | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [2]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        eval_cache = fem_dir / "eval_ref_T.npz"
        if eval_cache.is_file() and eval_cache.stat().st_mtime >= fem_json.stat().st_mtime:
            try:
                cached_T = np.asarray(np.load(eval_cache)["T"], dtype=float)
                if cached_T.shape == (len(xy),) and np.isfinite(cached_T).all():
                    return cached_T, f"FEM_WSL_cache:{case_id}"
            except Exception:
                pass
        art = SolutionArtifact.load(fem_json)
        ref_T = np.asarray(art.predict(xy)["T"], dtype=float)
        try:
            np.savez_compressed(eval_cache, T=ref_T)
        except OSError:
            pass
        return ref_T, f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
promo_file = CAMP_WS / "promoted_models.json"
if promo_file.is_file():
    promo_data = json.loads(promo_file.read_text(encoding="utf-8-sig"))
    PROMOTED = promo_data["promoted_models"]
    PROMOTED_ENERGY = promo_data.get("promoted_energy_config", {})
else:
    PROMOTED = ["M1", "M1_EXACT", "CUSPR", "M0R", "M2"]
    PROMOTED_ENERGY = {}

DEFAULT_ENERGY_CFG = {"energy_mode": "with_energy", "use_energy_loss": True, "energy_weight": 100.0, "exact_energy": True}
DIVERGENCE_STOP_S3 = {
    "max_loss": 1e12,
    "explosion_factor": 1e4,
    "patience_explosion": 3,
    "growth_patience": 6,
    "min_growth_ratio": 1.05,
    "warmup_steps": 150,
    "lbfgs_explosion_factor": 100.0,
    "lbfgs_stagnation_checks": 8,
    "check_nonfinite_grads": True,
}

SEEDS = (0,) if RUN_MODE == "demo_fast" else (0, 1, 2)
CASE_IDS_S3 = (
    ["mms_constant", "mms_layered_y", "cable_disk", "xlpe_single", "xlpe_discrete_layers", "aras_flat", "coaxial_robin"]
    if RUN_MODE == "demo_fast"
    else [e["id"] for e in catalog.list_entries()]
)
print("Modelos promovidos:", PROMOTED, "| Config energy:", PROMOTED_ENERGY, "| Casos:", len(CASE_IDS_S3), "| Semillas:", SEEDS)

Modelos promovidos: ['M0R', 'M1', 'FBPINN', 'M2', 'DCSNN'] | Config energy: {'DCSNN': {'energy_mode': 'with_energy', 'energy_weight': 100.0, 'exact_energy': True, 'use_energy_loss': True}, 'FBPINN': {'energy_mode': 'with_energy', 'energy_weight': 100.0, 'exact_energy': True, 'use_energy_loss': True}, 'M0R': {'energy_mode': 'with_energy', 'energy_weight': 100.0, 'exact_energy': True, 'use_energy_loss': True}, 'M1': {'energy_mode': 'with_energy', 'energy_weight': 100.0, 'exact_energy': True, 'use_energy_loss': True}, 'M2': {'energy_mode': 'with_energy', 'energy_weight': 100.0, 'exact_energy': True, 'use_energy_loss': True}} | Casos: 29 | Semillas: (0, 1, 2)


In [ ]:
stage3_dir = CAMP_WS / "stage3_multicase_smoke"
stage3_dir.mkdir(parents=True, exist_ok=True)
summary_path = stage3_dir / "summary.json"
summary = {} if OVERWRITE else (json.loads(summary_path.read_text(encoding="utf-8-sig")) if summary_path.is_file() else {})
tasks_s3 = []
task_meta_s3 = {}
reused_s3 = 0
dirty_s3 = False
case_cache_s3 = {}

def _s3_line_metrics(rec: dict) -> str:
    t_s = f"{rec['elapsed_s']:.1f}s" if rec.get("elapsed_s") is not None else "-"
    return f"NRMSE={rec.get('nrmse_pct')} | dTmax={rec.get('error_Tmax_abs_K')} K | bal={rec.get('balance_pct')}% | t={t_s}"

for case_id in CASE_IDS_S3:
    for arch in PROMOTED if RUN_MODE == "full_campaign" else PROMOTED[:3]:
        e_cfg = PROMOTED_ENERGY.get(arch, DEFAULT_ENERGY_CFG)
        e_weight = float(e_cfg.get("energy_weight", 100.0))
        e_exact = bool(e_cfg.get("exact_energy", e_cfg.get("use_energy_loss", True)))
        for seed in SEEDS:
            key = f"{case_id}:{arch}:seed{seed}"
            out_dir = stage3_dir / case_id / arch / f"seed_{seed}"
            if not OVERWRITE:
                cached = summary.get(key)
                if cached is not None and cached.get("status") in ("completed", "diverged"):
                    reused_s3 += 1
                    print(f"[Etapa 3 | CACHED] {key:36s} | status={cached.get('status')} | {_s3_line_metrics(cached)}", flush=True)
                    continue
                res_file = out_dir / "result.json"
                if res_file.is_file():
                    res_disk = json.loads(res_file.read_text(encoding="utf-8-sig"))
                    if res_disk.get("status") in ("completed", "diverged"):
                        m = extract_metrics(res_disk)
                        ref_lbl_disk = (res_disk.get("evaluation") or {}).get("reference_label") or "disk_cache"
                        summary[key] = {
                            "case": case_id,
                            "architecture": arch,
                            "seed": seed,
                            "energy_mode": e_cfg.get("energy_mode", "with_energy"),
                            "use_energy_loss": bool(e_cfg.get("use_energy_loss", True)),
                            "energy_weight": e_weight,
                            "status": res_disk.get("status"),
                            "divergence_stop": res_disk.get("divergence_stop"),
                            "reference_label": ref_lbl_disk,
                            "smoke_only": True,
                            **m,
                        }
                        if res_disk.get("error"):
                            summary[key]["error"] = res_disk.get("error")
                        dirty_s3 = True
                        reused_s3 += 1
                        print(f"[Etapa 3 | CACHED result.json] {key:36s} | status={summary[key]['status']} | {_s3_line_metrics(summary[key])}", flush=True)
                        continue
            if case_id not in case_cache_s3:
                t_prep = time.perf_counter()
                print(f"[Etapa 3 | Prep] ({len(case_cache_s3) + 1}/{len(CASE_IDS_S3)}) Cargando referencia {case_id}...", flush=True)
                prob = catalog.load(case_id)
                ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
                case_cache_s3[case_id] = (prob, ref_vals, ref_lbl)
                print(f"[Etapa 3 | Prep] {case_id} listo ({ref_lbl}) en {time.perf_counter() - t_prep:.2f}s", flush=True)
            prob, ref_vals, ref_lbl = case_cache_s3[case_id]
            task_meta_s3[key] = (case_id, arch, seed, ref_lbl, e_cfg)
            tasks_s3.append(
                RunTask(
                    name=key,
                    problem_source=prob.doc,
                    architecture=arch,
                    output_dir=out_dir,
                    config_kwargs={
                        "adam_steps": 60 if RUN_MODE == "demo_fast" else 800,
                        "lbfgs_steps": 5 if RUN_MODE == "demo_fast" else 50,
                        "lbfgs_iterations": 1,
                        "lr": 0.002,
                        "width": 32,
                        "depth": 3,
                        "dtype": "float32",
                        "device": DEVICE,
                        "init_seed": seed,
                        "sampling_seed": 11 + seed,
                        "weights": {"pde": 1.0, "energy": e_weight},
                        "exact_energy": e_exact,
                        "divergence_stop": DIVERGENCE_STOP_S3,
                        "points": {"n_interior": 512, "n_layer": 256, "n_face": 128, "n_boundary": 128},
                        "sampling": "mixed",
                        "focused": True,
                        "focus_criterion": "residual",
                        "focus_surfaces": False,
                        "resample_every": 100,
                    },
                    reference=ref_vals,
                    reference_label=ref_lbl,
                    overwrite=True,
                )
            )

def _on_stage3_done(res: dict) -> None:
    key = res["name"]
    case_id, arch, seed, ref_lbl, e_cfg = task_meta_s3[key]
    summary[key] = {
        "case": case_id,
        "architecture": arch,
        "seed": seed,
        "energy_mode": e_cfg.get("energy_mode", "with_energy"),
        "use_energy_loss": bool(e_cfg.get("use_energy_loss", True)),
        "energy_weight": float(e_cfg.get("energy_weight", 100.0)),
        "status": res.get("status"),
        "divergence_stop": res.get("divergence_stop"),
        "reference_label": ref_lbl,
        "smoke_only": True,
        **extract_metrics(res),
    }
    if res.get("error"):
        summary[key]["error"] = res.get("error")
    write_json_atomic(summary_path, summary)
    print(f"[Etapa 3] {key:36s} | dev={res.get('device')} | status={summary[key]['status']} | {_s3_line_metrics(summary[key])}", flush=True)

if tasks_s3:
    n_workers_s3 = MAX_WORKERS if USE_PARALLEL else 1
    print(f"[Etapa 3] Iniciando {len(tasks_s3)} corridas nuevas (reutilizadas={reused_s3}, workers={n_workers_s3})...", flush=True)
    runner_s3 = ParallelRunner(max_workers=n_workers_s3, distribute_devices=not USE_PARALLEL)
    runner_s3.run(tasks_s3, on_complete=_on_stage3_done, show_progress=True, progress_title="Etapa 3 — Smoke Multicaso")
    write_json_atomic(summary_path, summary)
else:
    if dirty_s3:
        write_json_atomic(summary_path, summary)
    print(f"[Etapa 3] Todas las {reused_s3} corridas ya estaban guardadas en disco (OVERWRITE=False); no se re-entrenó nada.")
summary_path

[Etapa 3 | Prep] (1/29) Cargando referencia annulus...
[Etapa 3 | Prep] annulus listo (analytic_reference) en 0.03s
[Etapa 3 | Prep] (2/29) Cargando referencia cable_disk...
[Etapa 3 | Prep] cable_disk listo (FEM_WSL_cache:cable_disk) en 0.04s
[Etapa 3 | Prep] (3/29) Cargando referencia mms_constant...
[Etapa 3 | Prep] mms_constant listo (analytic_reference) en 0.06s
[Etapa 3 | Prep] (4/29) Cargando referencia mms_high_contrast...
[Etapa 3 | Prep] mms_high_contrast listo (analytic_reference) en 0.05s
[Etapa 3 | Prep] (5/29) Cargando referencia mms_interface...
[Etapa 3 | Prep] mms_interface listo (analytic_reference) en 0.04s
[Etapa 3 | Prep] (6/29) Cargando referencia mms_layered_y...
[Etapa 3 | Prep] mms_layered_y listo (analytic_reference) en 0.04s
[Etapa 3 | Prep] (7/29) Cargando referencia mms_robin...
[Etapa 3 | Prep] mms_robin listo (analytic_reference) en 0.03s
[Etapa 3 | Prep] (8/29) Cargando referencia mms_sine...
[Etapa 3 | Prep] mms_sine listo (analytic_reference) en 0.02s


Tarea,Estado,Fase,Pasos,Progreso,%,Loss,Tiempo
kim_2024_thermal_original:DCSNN:seed0,running,adam,1 / 850,,0.1%,1.88e+04,95.9s
kim_2024_thermal_original:DCSNN:seed1,running,adam,1 / 850,,0.1%,9.34e+03,94.2s
kim_2024_thermal_original:DCSNN:seed2,running,adam,1 / 850,,0.1%,6.26e+04,92.8s
kim_layered:M0R:seed0,running,adam,1 / 850,,0.1%,3.60e+04,9.7s
kim_layered:M0R:seed1,running,adam,1 / 850,,0.1%,1.29e+04,8.7s
kim_layered:M0R:seed2,running,adam,1 / 850,,0.1%,2.16e+04,9.0s
kim_2024_thermal_original:M0R:seed2,completed,done,850 / 850,,100.0%,2.59e+01,3616.8s
kim_2024_thermal_original:M1:seed0,completed,done,850 / 850,,100.0%,3.40e+01,1553.7s
kim_2024_thermal_original:M1:seed1,completed,done,850 / 850,,100.0%,2.56e+01,1544.3s
kim_2024_thermal_original:M1:seed2,completed,done,850 / 850,,100.0%,3.79e+01,1540.5s


C:\Users\QU1267\AppData\Local\Temp\ipykernel_9300\122547822.py:121: RuntimeWarning: El pool de 12 procesos se interrumpió (BrokenProcessPool); reintentando 435 tarea(s) pendientes con 6 trabajador(es).
  runner_s3.run(tasks_s3, on_complete=_on_stage3_done, show_progress=True, progress_title="Etapa 3 — Smoke Multicaso")


[Etapa 3] annulus:M1:seed0                     | dev=cpu | status=completed | NRMSE=597.1017774276824 | dTmax=76.77482182915028 K | bal=38.202926988708306% | t=33.1s
[Etapa 3] annulus:M1:seed1                     | dev=cpu | status=completed | NRMSE=999.9003342452023 | dTmax=193.9006419890624 K | bal=115.34222747777837% | t=34.7s
[Etapa 3] annulus:M1:seed2                     | dev=cpu | status=completed | NRMSE=749.5386455925596 | dTmax=112.40377767021474 K | bal=101.038164894845% | t=34.7s
[Etapa 3] annulus:M0R:seed2                    | dev=cpu | status=completed | NRMSE=594.0950288932028 | dTmax=50.571487020800674 K | bal=201.53685032149448% | t=39.1s
[Etapa 3] annulus:M0R:seed0                    | dev=cpu | status=completed | NRMSE=672.0564033216317 | dTmax=101.5949550383788 K | bal=200.9046077031122% | t=39.2s
[Etapa 3] annulus:M0R:seed1                    | dev=cpu | status=completed | NRMSE=813.3961882596432 | dTmax=105.01858479912099 K | bal=39.2313174168147% | t=40.5s
[Etapa

In [ ]:
import pandas as pd

df_s3 = pd.DataFrame(summary).T
df_s3 = df_s3[df_s3["status"] == "completed"].copy()
num_cols = ["error_Tmax_abs_K", "nrmse_pct", "balance_pct"]
df_s3[num_cols] = df_s3[num_cols].astype(float)

def cumple(row, crit):
    return all(row[k] <= crit[k] for k in crit)

df_s3["cumple_estricto"] = df_s3.apply(cumple, axis=1, crit=STRICT_CRITERIA)
df_s3["cumple_relajado"] = df_s3.apply(cumple, axis=1, crit=RELAXED_CRITERIA)

ranking = (
    df_s3.groupby("architecture")
    .agg(
        casos=("case", "count"),
        nrmse_medio=("nrmse_pct", "mean"),
        err_Tmax_medio_K=("error_Tmax_abs_K", "mean"),
        balance_medio_pct=("balance_pct", "mean"),
        cumple_relajado=("cumple_relajado", "sum"),
    )
    .sort_values("nrmse_medio")
)
ranking.insert(0, "rank", range(1, len(ranking) + 1))
print(ranking.round(3))

mejor_por_caso = df_s3.loc[
    df_s3.groupby("case")["nrmse_pct"].idxmin(),
    ["case", "architecture", "nrmse_pct"],
]
print("\nMejor modelo por caso (NRMSE):")
print(mejor_por_caso.to_string(index=False))